# Phase 6 V3-tiling — Patch-Based Texture Expansion (Single Stage, V2 Recipe)

**Goal:** push the texture head from V2's 67.86% / F1 0.678 toward 78–85%
by expanding the texture training signal via 3×3 tiling of source images
(2,007 / 252 / 252 train/val/test patches versus the 223 / 28 / 28 source
images V2 saw). Soil_type and moisture must each stay within 2 pts of V2.

## What this notebook is — and is not

- **IS:** the V2 training recipe **unchanged**. Same backbone, optimizer,
  scheduler, augmentation pipeline, Mixup/CutMix, label smoothing, epoch
  count. Only the texture dataset is swapped to the tiled one.
- **IS NOT:** a staged or sequential training schedule of any kind. An
  earlier 3-stage 'V3 sequential transfer' attempt collapsed all heads to
  ~20% accuracy and is **ABANDONED**. Do not re-introduce that pattern.

## Epoch-1 health check

Right after the FIRST epoch's val pass, Cell 9 calls
`health_check(val_metrics)` — if any head's val top-1 is below 40%, the
run aborts immediately rather than burning the remaining ~10 hours of
Colab time on a doomed trajectory.

## V2 baseline (TTA), the bar to beat

- `soil_type`: 89.92% / F1 0.851
- `moisture`:  95.76% / F1 0.958
- `texture`:   67.86% / F1 0.678

**Ship criteria:** texture top-1 strictly improves AND soil_type and
moisture each stay within 2 pts of V2. Otherwise V2 remains production
(its repo `ankit-iiitdmj/iks-soil-multitask-v2` is untouched).


In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

import os, subprocess, sys

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "requirements.txt")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

os.chdir(REPO_PATH)
print("Repo root contents:", sorted(os.listdir(".")))

_pip_packages = [
    "timm>=1.0.0",
    "albumentations>=1.4",
    "huggingface_hub>=0.24",
    "datasets>=2.20",
    "iterative-stratification>=0.1.7",
    "pydantic>=2.7",
    "pyyaml>=6.0",
]
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", *_pip_packages],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print("PIP STDOUT (tail):\n" + proc.stdout[-3000:])
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tails above.")
print("setup ok")


In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
print(f"HF Hub ok: user={_whoami['name']}")


In [ ]:
# Cell 4 — GPU check + auto batch size
import subprocess, torch, sys
sys.path.insert(0, REPO_PATH)

subprocess.run(["nvidia-smi"], check=False)
print()
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"GPU: {dev.name}, VRAM: {dev.total_memory / 1024**3:.1f} GiB")
else:
    print("No GPU detected — switch runtime to GPU before continuing.")

from src.soil.train import auto_batch_size
batch = auto_batch_size()


## V3-tiling Configuration (identical to V2 except texture dataset)

- Backbone: **EfficientNet-B0 @ 224×224** (UNCHANGED from V1/V2)
- 40 epochs (5 frozen warmup + 35 unfrozen) — same as V2
- AdamW `lr=1e-4`, `weight_decay=1e-4`; CosineAnnealingLR `T_max=40`
- V2 strong augmentation (`build_soil_train_aug_v2`)
- Mixup/CutMix `p=0.3 / α=0.2 / cutmix α=1.0`
- Label smoothing 0.1 in `compute_multitask_loss_smoothed`
- TTA (5 views) on the final test eval — same as V2 Cell 11
- **Texture dataset:** `ankit-iiitdmj/iks-soil-texture-tiled` (NEW, 2,511 patches)
- Per-epoch checkpoint pushed to `ankit-iiitdmj/iks-soil-multitask-v3-tiling`
- **Epoch-1 health check:** abort if any head val top-1 < 40%


In [ ]:
# Cell 6 — Load 3 HF datasets. Texture is the NEW tiled dataset.
from datasets import load_dataset
from src.soil.train_v3_tiling import HF_DATASETS_V3_TILING

DATASETS = HF_DATASETS_V3_TILING
print("Datasets (note texture swap vs V2):")
for head, repo in DATASETS.items():
    print(f"  {head}: {repo}")

loaded = {head: load_dataset(repo) for head, repo in DATASETS.items()}
for head, dsd in loaded.items():
    repo = DATASETS[head].split("/")[-1]
    print(f"{repo}: train={len(dsd['train'])} val={len(dsd['val'])} test={len(dsd['test'])}")


In [ ]:
# Cell 7 — V2 transforms + per-task DataLoaders (UNCHANGED from V2, only the texture rows change)
import numpy as np
import torch
import yaml
from torch.utils.data import ConcatDataset, DataLoader, Dataset

from src.soil.dataset import build_multitask_labels
from src.soil.transforms import build_soil_eval_aug
from src.soil.transforms_v2 import build_soil_train_aug_v2

with open("configs/data/soil_norm.yaml") as fh:
    norm = yaml.safe_load(fh)
MEAN = tuple(norm["mean"])
STD  = tuple(norm["std"])
IMG_SIZE = 224

train_aug = build_soil_train_aug_v2(IMG_SIZE, MEAN, STD)
eval_aug  = build_soil_eval_aug(IMG_SIZE, MEAN, STD)


class _MultiTaskWrapper(Dataset):
    """Wrap an HF dataset split as multi-task rows (one head valid, others -1)."""

    def __init__(self, hf_split, head: str, transform):
        self.hf_split = hf_split
        self.head = head
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        row = self.hf_split[idx]
        arr = np.asarray(row["image"].convert("RGB"))
        tensor = self.transform(image=arr)["image"]
        labels = build_multitask_labels(
            self.head, int(row["label_idx"]),
            head_order=("soil_type", "moisture", "texture"),
        )
        return {
            "image": tensor,
            "soil_type_label": torch.tensor(labels["soil_type"], dtype=torch.long),
            "moisture_label":  torch.tensor(labels["moisture"],  dtype=torch.long),
            "texture_label":   torch.tensor(labels["texture"],   dtype=torch.long),
        }


train_datasets = [
    _MultiTaskWrapper(loaded["soil_type"]["train"], "soil_type", train_aug),
    _MultiTaskWrapper(loaded["moisture"]["train"],  "moisture",  train_aug),
    _MultiTaskWrapper(loaded["texture"]["train"],   "texture",   train_aug),
]
train_loader = DataLoader(
    ConcatDataset(train_datasets),
    batch_size=batch, shuffle=True, num_workers=2, pin_memory=True,
)

val_loaders = {
    head: DataLoader(
        _MultiTaskWrapper(loaded[head]["val"], head, eval_aug),
        batch_size=batch, shuffle=False, num_workers=2, pin_memory=True,
    )
    for head in ("soil_type", "moisture", "texture")
}

print(f"train loader: {len(train_loader.dataset)} samples in {len(train_loader)} batches")
for head in ("soil_type", "moisture", "texture"):
    print(f"  val {head}: {len(val_loaders[head].dataset)} samples")


In [ ]:
# Cell 8 — build model + optimizer + scheduler + scaler + V3-tiling ckpt mgr
import torch
from src.soil.model import SoilMultiTaskClassifier
from src.soil.train_v3_tiling import SoilCheckpointManagerV3Tiling

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SoilMultiTaskClassifier(
    backbone_name="efficientnet_b0", pretrained=True, dropout=0.3,
).to(device)

EPOCHS_TOTAL = 40
FREEZE_EPOCHS = 5
MIX_P = 0.3
LABEL_SMOOTHING = 0.1

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_TOTAL)
scaler = torch.amp.GradScaler("cuda")
ckpt_mgr = SoilCheckpointManagerV3Tiling()

start_epoch = 0
history: list[dict] = []
best_val_sum = 0.0

_resume = ckpt_mgr.try_load_latest()
if _resume is not None:
    print(f"Resuming V3-tiling from epoch {_resume['epoch']} ...")
    model.load_state_dict(_resume["model_state"])
    if _resume.get("optimizer_state") is not None: optimizer.load_state_dict(_resume["optimizer_state"])
    if _resume.get("scheduler_state") is not None: scheduler.load_state_dict(_resume["scheduler_state"])
    if _resume.get("scaler_state") is not None: scaler.load_state_dict(_resume["scaler_state"])
    start_epoch = int(_resume["epoch"]) + 1
    history = list(_resume.get("history", []))
    if history:
        best_val_sum = max(h.get("val_sum", 0.0) for h in history)
    print(f"  history len={len(history)} best_val_sum={best_val_sum:.4f}")
else:
    print("No prior V3-tiling checkpoint — starting fresh from ImageNet weights.")


In [ ]:
# Cell 9 — V2-recipe training loop, single stage, with epoch-1 health check
import time, datetime
from src.soil.train import evaluate_per_task
from src.soil.train_v2 import train_one_epoch_v2
from src.soil.train_v3_tiling import health_check

for epoch in range(start_epoch, EPOCHS_TOTAL):
    is_frozen = epoch < FREEZE_EPOCHS
    if is_frozen:
        model.freeze_backbone()
        stage_tag = "FROZEN"
    else:
        model.unfreeze_backbone()
        stage_tag = "UNFROZEN"

    t0 = time.time()
    train_losses = train_one_epoch_v2(
        model, train_loader, optimizer, scaler, device,
        mix_p=MIX_P, label_smoothing=LABEL_SMOOTHING,
    )
    val_metrics = evaluate_per_task(model, val_loaders, device)
    scheduler.step()
    elapsed = time.time() - t0

    val_sum = sum(val_metrics[h]["top1_accuracy"] for h in ("soil_type", "moisture", "texture"))

    print(
        f"[V3-tiling epoch {epoch+1}/{EPOCHS_TOTAL}] {stage_tag} | "
        f"type_loss={train_losses['loss_soil_type']:.4f} "
        f"moist_loss={train_losses['loss_moisture']:.4f} "
        f"tex_loss={train_losses['loss_texture']:.4f} "
        f"total={train_losses['loss_total']:.4f} | "
        f"val: type_acc={val_metrics['soil_type']['top1_accuracy']:.4f} "
        f"moist_acc={val_metrics['moisture']['top1_accuracy']:.4f} "
        f"tex_acc={val_metrics['texture']['top1_accuracy']:.4f} | "
        f"{elapsed:.0f}s"
    )

    # Epoch-1 (the first epoch of THIS run, even after a resume) health check.
    if epoch == start_epoch:
        health_check(val_metrics)

    history.append({
        "epoch": int(epoch + 1),
        "stage": stage_tag,
        **train_losses,
        "val_soil_type": val_metrics["soil_type"],
        "val_moisture":  val_metrics["moisture"],
        "val_texture":   val_metrics["texture"],
        "val_sum": float(val_sum),
        "elapsed_seconds": float(elapsed),
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    })

    ckpt_mgr.save(
        epoch=epoch, model_state=model.state_dict(),
        optimizer_state=optimizer.state_dict(),
        scheduler_state=scheduler.state_dict(),
        scaler_state=scaler.state_dict(),
        history=history, val_metrics=val_metrics,
    )

    if val_sum > best_val_sum:
        best_val_sum = float(val_sum)
        ckpt_mgr.save_best(epoch=epoch, model_state=model.state_dict(), val_metrics=val_metrics)
        print(f"  -> new best (val_sum={best_val_sum:.4f}) checkpoint_best.pt pushed")

best_epoch = max(history, key=lambda h: h["val_sum"]) if history else None
if best_epoch:
    print(
        f"\nV3-tiling training complete. best epoch={best_epoch['epoch']} "
        f"val_sum={best_epoch['val_sum']:.4f}"
    )


## Final test evaluation with TTA

Mirrors V2's Cell 11 exactly. Loads the V3-tiling best checkpoint,
runs 5-view TTA on each task's held-out test split (texture test split
is the **tiled** test patches — from source images never seen during
training, leakage-safe by construction), and pushes
`eval_metrics_test.json` to `ankit-iiitdmj/iks-soil-multitask-v3-tiling`.

V2 baseline (TTA) for reference:
- `soil_type`: 89.92% / F1 0.851
- `moisture`:  95.76% / F1 0.958
- `texture`:   67.86% / F1 0.678


In [ ]:
# Cell 11 — TTA test eval (patch-level for all three tasks)
import json, torch
from huggingface_hub import HfApi
from torch.utils.data import DataLoader, Dataset

from src.soil.transforms_v2 import build_tta_views
from src.soil.train_v2 import evaluate_per_task_tta

_best = ckpt_mgr.try_load_best()
if _best is None:
    print("checkpoint_best.pt missing — falling back to checkpoint_latest.pt.")
    _best = ckpt_mgr.try_load_latest()
if _best is None:
    raise RuntimeError("No V3-tiling checkpoint on HF Hub — run Cell 9 first.")
model.load_state_dict(_best["model_state"])
model.eval()
print(f"Loaded V3-tiling checkpoint (epoch {_best['epoch']})")

tta_views = build_tta_views(IMG_SIZE, MEAN, STD)


class _RawTestDataset(Dataset):
    def __init__(self, hf_split, return_source_id=False):
        self.hf_split = hf_split
        self.return_source_id = return_source_id
    def __len__(self):
        return len(self.hf_split)
    def __getitem__(self, idx):
        row = self.hf_split[idx]
        arr = np.asarray(row["image"].convert("RGB"))
        if self.return_source_id:
            return arr, int(row["label_idx"]), str(row.get("source_id", f"row_{idx}"))
        return arr, int(row["label_idx"])


def _raw_collate(samples):
    return [s[0] for s in samples], torch.tensor([s[1] for s in samples], dtype=torch.long)


tta_loaders = {
    head: DataLoader(
        _RawTestDataset(loaded[head]["test"]),
        batch_size=batch, shuffle=False, num_workers=0, collate_fn=_raw_collate,
    )
    for head in ("soil_type", "moisture", "texture")
}
patch_metrics = evaluate_per_task_tta(model, tta_loaders, tta_views, device)
print("\n=== Patch-level TTA metrics ===")
for head, m in patch_metrics.items():
    print(f"  {head}: top1={m['top1_accuracy']:.4f} F1={m['macro_f1']:.4f} n={m['n_samples']}")


In [ ]:
# Cell 12 — image-level texture eval (majority vote over patches) + V2 vs V3-tiling verdict
import json, torch
from collections import Counter
from huggingface_hub import HfApi
from sklearn.metrics import f1_score

# Image-level eval: vote each test source image's patches
tex_source_preds: dict[str, list[int]] = {}
tex_source_label: dict[str, int] = {}

tex_raw = _RawTestDataset(loaded["texture"]["test"], return_source_id=True)

def _raw_collate_with_sid(samples):
    return [s[0] for s in samples], torch.tensor([s[1] for s in samples], dtype=torch.long), [s[2] for s in samples]

loader = DataLoader(tex_raw, batch_size=batch, shuffle=False, num_workers=0, collate_fn=_raw_collate_with_sid)

with torch.no_grad():
    for batch_imgs, batch_labels, batch_sids in loader:
        # 5-view TTA averaging
        view_logits = []
        for view in tta_views:
            tensors = torch.stack([view(image=img)["image"] for img in batch_imgs]).to(device)
            view_logits.append(model(tensors)["texture"])
        avg_logits = torch.stack(view_logits, dim=0).mean(dim=0)
        preds = avg_logits.argmax(dim=1).tolist()
        for sid, pred, label in zip(batch_sids, preds, batch_labels.tolist()):
            tex_source_preds.setdefault(sid, []).append(int(pred))
            tex_source_label.setdefault(sid, int(label))

image_y_true: list[int] = []
image_y_pred: list[int] = []
for sid, preds in tex_source_preds.items():
    voted = Counter(preds).most_common(1)[0][0]
    image_y_pred.append(voted)
    image_y_true.append(tex_source_label[sid])
image_top1 = sum(1 for t, p in zip(image_y_true, image_y_pred) if t == p) / max(1, len(image_y_true))
image_f1 = float(f1_score(image_y_true, image_y_pred, average="macro", zero_division=0.0))

print("=== Image-level texture (majority vote over patches per source image) ===")
print(f"  top1={image_top1:.4f}  F1={image_f1:.4f}  n_images={len(image_y_true)}")

summary = {
    head: {
        "top1_accuracy": patch_metrics[head]["top1_accuracy"],
        "top5_accuracy": patch_metrics[head]["top5_accuracy"],
        "macro_f1":      patch_metrics[head]["macro_f1"],
        "n_samples":     patch_metrics[head]["n_samples"],
        "num_classes":   {"soil_type": 7, "moisture": 3, "texture": 3}[head],
    }
    for head in ("soil_type", "moisture", "texture")
}
summary["texture_image_level"] = {
    "top1_accuracy": image_top1, "macro_f1": image_f1, "n_images": len(image_y_true),
}
tmp_path = "/tmp/eval_metrics_test.json"
with open(tmp_path, "w") as fh:
    json.dump(summary, fh, indent=2)
HfApi().upload_file(
    path_or_fileobj=tmp_path, path_in_repo="eval_metrics_test.json",
    repo_id=ckpt_mgr.repo_id, repo_type="model",
)
print("\nMetrics pushed to HF Hub.")

V2_BASELINE = {'soil_type': {'top1': 0.8992, 'f1': 0.8509}, 'moisture': {'top1': 0.9576, 'f1': 0.9578}, 'texture': {'top1': 0.6786, 'f1': 0.6778}}
print()
print("COMPARISON V2 vs V3-tiling (both with TTA):")
print("                              V2          V3-tiling     Δ")
deltas = {}
for head in ("soil_type", "moisture", "texture"):
    v2_top1 = V2_BASELINE[head]["top1"]; v2_f1 = V2_BASELINE[head]["f1"]
    v3_top1 = patch_metrics[head]["top1_accuracy"]; v3_f1 = patch_metrics[head]["macro_f1"]
    deltas[head] = (v3_top1 - v2_top1) * 100
    print(f"  {head:8s} top-1 (patch):  {v2_top1*100:5.2f}%      {v3_top1*100:5.2f}%      {(v3_top1-v2_top1)*100:+5.2f} pts")
    print(f"  {head:8s} F1    (patch):  {v2_f1:.3f}       {v3_f1:.3f}        {(v3_f1-v2_f1):+.3f}")
print(f"  texture top-1 (image-vote): {V2_BASELINE['texture']['top1']*100:5.2f}%      {image_top1*100:5.2f}%      {(image_top1-V2_BASELINE['texture']['top1'])*100:+5.2f} pts")

# Ship criteria: texture improved AND soil_type / moisture within 2 pts of V2.
tex_gain = deltas["texture"]
soil_drop = -deltas["soil_type"]
moist_drop = -deltas["moisture"]
ship = (tex_gain > 0.0) and (soil_drop <= 2.0) and (moist_drop <= 2.0)
print()
print(f"texture gain: {tex_gain:+.2f} pts (need > 0)")
print(f"soil_type change: {deltas['soil_type']:+.2f} pts (need >= -2.00)")
print(f"moisture change: {deltas['moisture']:+.2f} pts (need >= -2.00)")
print()
print("VERDICT:", "SHIP V3-TILING" if ship else "KEEP V2")


## V3-tiling complete

Final model checkpoint: `ankit-iiitdmj/iks-soil-multitask-v3-tiling` (private).
V1 (`iks-soil-multitask`) and V2 (`iks-soil-multitask-v2`) are untouched.

### Decision tree

- **VERDICT: SHIP V3-TILING** — texture improved AND soil_type/moisture
  stayed within 2 pts of V2. Use V3-tiling as the production soil model
  in Phase 8 integration. Note **patch-level** and **image-level**
  texture accuracies separately in the paper — patch-level is a fair
  metric (test patches come from held-out source images) but the
  image-vote is closer to deployment behaviour.
- **VERDICT: KEEP V2** — V3-tiling didn't clear the bar. V3-tiling
  stays on HF Hub as a documented negative result (useful for the
  ablation table: tiling alone wasn't enough on this dataset). V2
  remains production.

### Why this approach is safer than the abandoned 3-stage V3

- Single-stage training — no optimizer/scheduler resets mid-run.
- All heads are supervised the same way V2 supervised them — no head
  is left at random init partway through.
- Source-image-level split integrity is enforced at dataset build time
  (Part 1 script's disjointness assertion + audit JSON), so test
  patches genuinely come from photographs that were never seen during
  training.
- Epoch-1 health check ensures any collapse aborts within minutes,
  not after 10+ hours.
